In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project Root:", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)

Project Root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Data Directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\data\raw


In [2]:
cards = pd.read_csv(DATA_DIR / "EN_Card_Data.csv")

In [3]:
cards.shape

(2022, 17)

# Observation 1

The official English Pokémon TCG dataset contains:

- **2,022 cards**
- **17 attributes (columns)**

This dataset will serve as the primary knowledge base for our AI Training Agent.

In [6]:
cards.columns

Index(['Card ID', 'Card Name', 'Expansion', 'Collection No.',
       'Stage (Pokémon)/Type (Energy and Trainer)', 'Rule', 'Category',
       'Previous stage', 'HP', 'Type', 'Weakness', 'Resistance (Type)',
       'Retreat', 'Move Name', 'Cost', 'Damage', 'Effect Explanation'],
      dtype='str')

## Observation 2

The dataset contains 17 fields describing each Pokémon card.

Next, we will examine each field to determine how it contributes to AI decision-making, deck construction, and battle strategy.

In [8]:
cards.head()

,Card ID,Card Name,Expansion,Collection No.,Stage (Pokémon)/Type (Energy and Trainer),Rule,Category,Previous stage,HP,Type,Weakness,Resistance (Type),Retreat,Move Name,Cost,Damage,Effect Explanation
0,1,Basic {G} Energy,SVE,1,Basic Energy,NaN,NaN,NaN,NaN,{G},NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Basic {R} Energy,SVE,2,Basic Energy,NaN,NaN,NaN,NaN,{R},NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Basic {W} Energy,SVE,3,Basic Energy,NaN,NaN,NaN,NaN,{W},NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Basic {L} Energy,SVE,4,Basic Energy,NaN,NaN,NaN,NaN,{L},NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Basic {P} Energy,SVE,5,Basic Energy,NaN,NaN,NaN,NaN,{P},NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Observation 3

The first five records in the dataset are **Basic Energy** cards rather than Pokémon.

This indicates that the dataset stores **Pokémon, Trainer, and Energy cards together in one table**.

Fields that do not apply to a particular card type (such as HP or attacks for Energy cards) are represented as **NaN (Not a Number)** values.

In [9]:
cards["Category"].value_counts()

Category
Trainer's Pokémon（Team Rocket）    85
Tera(Stellar)                     60
Ancient                           26
Trainer's Pokémon（N）              26
Trainer's Pokémon（Hop）            21
Future                            17
Trainer's Pokémon（Ethan）          15
Trainer's Pokémon（Larry）          13
Trainer's Pokémon（Steven）         12
Trainer's Pokémon（Cynthia）        11
Trainer's Pokémon（Marnie）         11
Trainer's Pokémon（Erika）          11
Trainer's Pokémon（Misty）          10
Trainer's Pokémon（Arven）          10
Fossil                            10
Tera(Fighting)                     9
Tera(Lightning)                    9
Trainer's Pokémon（Iono）            9
Trainer's Pokémon（Lillie）          7
Tera(Fire)                         6
Tera(Darkness)                     3
Tera(Grass)                        3
Tera(Water)                        3
Tera(Dragon)                       3
Technical Machine                  2
Name: count, dtype: int64

## Observation 4

The `Category` column does **not** represent the broad card type (Pokémon, Trainer, or Energy).

Instead, it describes special classifications and mechanics such as:

- Trainer's Pokémon
- Ancient
- Future
- Tera Pokémon
- Fossil
- Technical Machine

This means we must use other columns to distinguish between Pokémon, Trainer, and Energy cards.

In [10]:
cards["Stage (Pokémon)/Type (Energy and Trainer)"].value_counts()

Stage (Pokémon)/Type (Energy and Trainer)
Basic Pokémon      958
Stage 1 Pokémon    618
Stage 2 Pokémon    229
Item                82
Supporter           61
Pokémon Tool        28
Stadium             26
Special Energy      12
Basic Energy         8
Name: count, dtype: int64

## Observation 5

The **Stage (Pokémon) / Type (Energy and Trainer)** column is one of the most important fields in the dataset.

It identifies the functional role of each card, including:

### Pokémon
- Basic Pokémon
- Stage 1 Pokémon
- Stage 2 Pokémon

### Trainer Cards
- Item
- Supporter
- Stadium
- Pokémon Tool

### Energy Cards
- Basic Energy
- Special Energy

This field will allow the AI to determine the legal actions available for each card during gameplay.

In [11]:
cards.isnull().sum()

Card ID                                         0
Card Name                                       0
Expansion                                      10
Collection No.                                  0
Stage (Pokémon)/Type (Energy and Trainer)       0
Rule                                         1669
Category                                     1630
Previous stage                               1165
HP                                            207
Type                                          197
Weakness                                      288
Resistance (Type)                            1646
Retreat                                       274
Move Name                                     211
Cost                                          466
Damage                                        737
Effect Explanation                            541
dtype: int64

## Observation 6

The dataset contains several columns with missing values.

These missing values are **expected** and are primarily caused by differences between Pokémon, Trainer, and Energy cards.

Examples include:

- Energy cards do not have HP or attacks.
- Trainer cards do not have retreat costs or weaknesses.
- Basic Pokémon do not have a previous evolution stage.

Therefore, these missing values represent **game mechanics**, not data quality issues.

In [12]:
cards.info()

<class 'pandas.DataFrame'>
RangeIndex: 2022 entries, 0 to 2021
Data columns (total 17 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Card ID                                    2022 non-null   int64  
 1   Card Name                                  2022 non-null   str    
 2   Expansion                                  2012 non-null   str    
 3   Collection No.                             2022 non-null   int64  
 4   Stage (Pokémon)/Type (Energy and Trainer)  2022 non-null   str    
 5   Rule                                       353 non-null    str    
 6   Category                                   392 non-null    str    
 7   Previous stage                             857 non-null    str    
 8   HP                                         1815 non-null   float64
 9   Type                                       1825 non-null   str    
 10  Weakness                           

## Observation 7

The dataset contains **2,022 records** and **17 columns**.

### Data Types

- **13 text (`str`) columns**
- **2 integer (`int64`) columns**
- **2 numeric (`float64`) columns**

The numeric columns (`HP` and `Retreat`) are stored as floating-point values because they contain missing values (`NaN`).

The entire dataset occupies only **268.7 KB** of memory, making it lightweight and efficient for AI processing.

In [13]:
cards[cards["Stage (Pokémon)/Type (Energy and Trainer)"] == "Basic Pokémon"].head()

,Card ID,Card Name,Expansion,Collection No.,Stage (Pokémon)/Type (Energy and Trainer),Rule,Category,Previous stage,HP,Type,Weakness,Resistance (Type),Retreat,Move Name,Cost,Damage,Effect Explanation
22,22,Hippopotas,DRI,105,Basic Pokémon,NaN,NaN,NaN,90.0,{F},{G},NaN,3.0,Push Down,{F},10,Switch out your opponent’s Active Pokémon to the Bench. (Your opponent chooses the new Active Pokémon.)
25,24,Team Rocket's Kangaskhan ex,ASC,162,Basic Pokémon,Pokémon ex,Trainer's Pokémon（Team Rocket）,NaN,230.0,{C},{F},NaN,2.0,\nComet Punch,●●,30×,Flip 4 coins. This attack does 30 damage for each heads.
26,24,Team Rocket's Kangaskhan ex,ASC,162,Basic Pokémon,Pokémon ex,Trainer's Pokémon（Team Rocket）,NaN,230.0,{C},{F},NaN,2.0,Wicked Impact,●●●,120,"If you played a Supporter card that has “Team Rocket” in its name from your hand during this turn, this attack does 100 more damage."
27,25,Pinsir,TWM,3,Basic Pokémon,NaN,NaN,NaN,110.0,{G},{R},NaN,2.0,Slow Crunch,{G}●,NaN,"Discard all Energy from this Pokémon. At the end of your opponent’s next turn, the Defending Pokémon will be Knocked Out."
28,25,Pinsir,TWM,3,Basic Pokémon,NaN,NaN,NaN,110.0,{G},{R},NaN,2.0,Superpowered Horns,{G}●●,100,NaN


## Observation 8

The dataset is organized with **one row per attack**, not one row per Pokémon card.

Cards that have multiple attacks appear multiple times.

For example:

- Team Rocket's Kangaskhan ex appears twice because it has two attacks.
- Pinsir appears twice because it has two attacks.

This design makes it easier for an AI agent to evaluate each available attack independently during battle.

In [14]:
cards[cards["Card Name"] == "Pinsir"]

,Card ID,Card Name,Expansion,Collection No.,Stage (Pokémon)/Type (Energy and Trainer),Rule,Category,Previous stage,HP,Type,Weakness,Resistance (Type),Retreat,Move Name,Cost,Damage,Effect Explanation
27,25,Pinsir,TWM,3,Basic Pokémon,NaN,NaN,NaN,110.0,{G},{R},NaN,2.0,Slow Crunch,{G}●,NaN,"Discard all Energy from this Pokémon. At the end of your opponent’s next turn, the Defending Pokémon will be Knocked Out."
28,25,Pinsir,TWM,3,Basic Pokémon,NaN,NaN,NaN,110.0,{G},{R},NaN,2.0,Superpowered Horns,{G}●●,100,NaN


## Observation 9

Cards are uniquely identified by **Card ID**.

Multiple rows may share the same Card ID because each row represents a different attack.

For example:

Card ID 25 (Pinsir)

- Slow Crunch
- Superpowered Horns

During battle, the AI should retrieve all rows with the same Card ID to determine every legal attack available for that Pokémon.

In [15]:
cards["Card ID"].nunique()

1267

## Observation 10

Although the dataset contains **2,022 rows**, there are only **1,267 unique Card IDs**.

This confirms that multiple rows may belong to the same card.

Each row represents one attack associated with that card.

Therefore:

- **Card ID** uniquely identifies a card.
- Multiple rows with the same Card ID represent different attacks.

The AI should organize information around **Card IDs**, not individual rows.

In [16]:
rows = len(cards)
unique_cards = cards["Card ID"].nunique()

print(f"Total rows: {rows}")
print(f"Unique cards: {unique_cards}")
print(f"Duplicate attack rows: {rows - unique_cards}")
print(f"Average rows per card: {rows / unique_cards:.2f}")


Total rows: 2022
Unique cards: 1267
Duplicate attack rows: 755
Average rows per card: 1.60


## Observation 11

The dataset contains:

- **2,022 total rows**
- **1,267 unique cards**
- **755 additional attack rows**

On average, each card has **1.60 attack records**.

This confirms that the dataset stores attacks separately rather than storing all attacks within a single row.

When building the AI, attack information should be grouped by **Card ID**.